In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

model=ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0.8,
    max_tokens=20
)

def determine_language(inputs):
    """判断语言种类"""
    query = inputs["query"]
    if "日语" in query:
        return "japanese"
    elif "韩语" in query:
        return "korean"
    else:
        return "english"

# 构建提示词
english_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个英语翻译专家，你叫小英"),
    ("human", "{query}")
])

japanese_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个日语翻译专家，你叫小日"),
    ("human", "{query}")
])

korean_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个韩语翻译专家，你叫小韩"),
    ("human", "{query}")
])
# 创建字符串输出解析器
parser = StrOutputParser()
# 构建链分支结构，默认分支为汉语
# 创建一个可运行的分支链，根据输入文本的语言类型选择相应的处理流程
# 该链会首先判断输入文本的语言，然后路由到对应的提示词模板、大语言模型和解析器组合

chain = RunnableBranch(
    (lambda x: determine_language(x) == "japanese", japanese_prompt | model | parser),
    (lambda x: determine_language(x) == "korean", korean_prompt | model | parser),
    (english_prompt | model | parser)
)

# 输出结果
print(f"输出结果：{chain.invoke({'query': '请你用韩语翻译这句话：“你好”'})}")

输出结果：“你好”用韩语翻译是“안녕하세요”。
